> 在线讲义：[第 64 章 · 模型服务的选择、成本与扩容：按成功任务算账](https://wikieden.github.io/ai-infra-from-constraints/lessons/64-service-cost.html) · 本书《从约束推导 AI Infra》

# 第 64 章 · 模型服务的选择、成本与扩容：按成功任务算账

配套讲义：仓库里 `ai-infra/notes/lessons/64-service-cost.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。



思考预算、前缀缓存计费、命中率与成功率的路由、自建 / 按量 / 订阅、提交粒度、恢复树与例 11-9 的方案选择，取自 AII 第 11 章 §11.4–§11.5。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `call_cost`**：C = (I_u p_u + I_w p_w + I_h p_h + O p_o) / 1e6，价格按美元/百万 token。

In [ ]:
def call_cost(inp: float, out: float, p_in: float, p_out: float, cached: float = 0, p_cache: float = 0, write: float = 0, p_write: float = 0) -> float:
    """C = (I_u p_u + I_w p_w + I_h p_h + O p_o) / 1e6，价格按美元/百万 token。"""
    return (inp * p_in + write * p_write + cached * p_cache + out * p_out) / 1e6

**函数 `cache_worth`**：n p_u > p_w + (n − 1) p_h。

In [ ]:
def cache_worth(n: int, pu: float, pw: float, ph: float) -> bool:
    """n p_u > p_w + (n − 1) p_h。"""
    return n * pu > pw + (n - 1) * ph

**函数 `breakeven_hit`**：令 B 的每成功成本等于 A，求 B 的命中率。

In [ ]:
def breakeven_hit(cA: float, pA: float, hit: float, miss: float, pB: float) -> float:
    """令 B 的每成功成本等于 A，求 B 的命中率。"""
    target = cA / pA * pB
    return (miss - target) / (miss - hit)

**函数 `selfhost_breakeven`**：小工具。

In [ ]:
def selfhost_breakeven(F: float, c: float, v: float = 0) -> float:
    return F / (c - v)

**函数 `commit_plan`**：(保存总 ms, 无故障总 s, 最多重做 s)。

In [ ]:
def commit_plan(n: int = 100, work_ms: float = 200, fixed: float = 8, per: float = 2, batch: int = 1):
    """(保存总 ms, 无故障总 s, 最多重做 s)。"""
    k = n // batch
    save = k * (fixed + batch * per)
    return save, (n * work_ms + save) / 1000, batch * work_ms / 1000

**函数 `recovery_tree`**：例 11-8：路径 (概率, 成本, 时间)。

In [ ]:
def recovery_tree():
    """例 11-8：路径 (概率, 成本, 时间)。"""
    return [(0.80, 0.010, 10), (0.12 * 0.6, 0.016, 14), (0.12 * 0.4 * 0.98, 0.046, 22), (0.12 * 0.4 * 0.02, 0.046, 22),
            (0.08 * 0.98, 0.040, 18), (0.08 * 0.02, 0.040, 18)]

**函数 `plan_cost`**：每项任务：模型（6.79 $/卡时）+ CPU（0.000014 $/CPU·秒）+ 内存（0.0000045 $/GiB·秒）。

In [ ]:
def plan_cost(call_b200_s: float, cpu_s: float, mem_gib_s: float, rounds: int = 3) -> float:
    """每项任务：模型（6.79 $/卡时）+ CPU（0.000014 $/CPU·秒）+ 内存（0.0000045 $/GiB·秒）。"""
    return rounds * call_b200_s * 6.79 / 3600 + cpu_s * 0.000014 + mem_gib_s * 0.0000045

**函数 `amdahl`**：小工具。

In [ ]:
def amdahl(f: float, s: float) -> float:
    return 1 / ((1 - f) + f / s)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：思考预算")
a, b = call_cost(10000, 1200, 2, 10), call_cost(10000, 300, 2, 10)
print(f"  思考 1000 vs 100：{a:.3f} vs {b:.3f} 美元（−{(1 - b / a) * 100:.0f}%）；100 次省 {100 * (a - b):.2f}、30 次重试花 {30 * a:.2f}")

In [ ]:
print("\n第二步：前缀缓存")
plain = 10 * call_cost(10000, 0, 2, 0)
cached = call_cost(2000, 0, 2, 0, write=8000, p_write=2.5) + 9 * call_cost(2000, 0, 2, 0, cached=8000, p_cache=0.2)
print(f"  十轮输入：无缓存 {plain:.2f}，有缓存 {cached:.4f} 美元；第 2 次就回本：{cache_worth(2, 2, 2.5, 0.2)}")
print("  请求命中率都 50%：只命中 1K 前缀读 10%，只命中 9K 前缀读 90%")

In [ ]:
print("\n第三步：路由")
print(f"  A 排 200 + 命中 50 + 网络 50 = 300 ms；B 20 + 500 + 50 = 570 ms")
cA = call_cost(1000, 2000, 1, 5, cached=19000, p_cache=0.1)
hit, miss = call_cost(1000, 300, 2, 10, cached=19000, p_cache=0.2), call_cost(20000, 300, 2, 10)
h = breakeven_hit(cA, 0.8, hit, miss, 0.98)
print(f"  A 每次 {cA:.4f}、成功 {cA / 0.8:.4f}；B 命中 {hit:.4f}、未命中 {miss:.4f} → 成本交点 h = {h:.1%}；6 s 期限要 h ≥ {0.90 / 0.98:.1%}")
print(f"  快速服务：16 会话 6 s/调用，180 并发 > 160 槽 → 12 个副本；每次 {4 * 9 / 32} → {4 * 6 / 16} B200·秒")

In [ ]:
print("\n第四步：自建 vs 按量")
F = 4 * 720 * 6.79
c = 27 * 4 / 32 * 8.64 / 3600
n = selfhost_breakeven(F, c)
cap = 32 * 2_592_000 / 27
print(f"  预留 {F:,.1f} 美元/月；按量每项 {c:.4f}；交点 {n / 1e4:.0f} 万项，上限 {cap / 1e4:.1f} 万项，利用率 {n / cap:.1%} = {6.79 / 8.64:.1%}")
print(f"  decode 快一倍：{15.9 / 2 + 9.4:.1f} ms/token，总加速 {25.3 / (15.9 / 2 + 9.4):.2f} 倍")

In [ ]:
print("\n第五步：恢复与提交")
for bt in (1, 10):
    s, t, r = commit_plan(batch=bt)
    print(f"  每 {bt} 个提交：保存 {s:.0f} ms，总 {t:.2f} s，最多重做 {r:.1f} s")
tree = recovery_tree()
cost = sum(p * c for p, c, _ in tree)
ok = sum(p for (p, _, _), good in zip(tree, (1, 1, 1, 0, 1, 0)) if good)
ontime = sum(p for (p, _, t), good in zip(tree, (1, 1, 1, 0, 1, 0)) if good and t <= 20)
print(f"  恢复树：每项 {cost:.5f} 美元，成功 {ok:.1%}，按时 {ontime:.1%}；每成功 {cost / ok:.4f}，每按时成功 {cost / ontime:.4f}（只做首次 0.0125）")

In [ ]:
print("\n第六步：例 11-9（每千项提交）")
for name, call, cpu, mem in (("普通 + 驻留", 1.125, 3, 60), ("普通 + 重建", 1.125, 3.3, 18), ("快速 + 驻留", 1.5, 3, 42), ("快速 + 重建", 1.5, 3.3, 18)):
    print(f"  {name}：{plan_cost(call, cpu, mem) * 1000:.2f} 美元")
print(f"  每千项按时成功：{plan_cost(1.5, 3.3, 18) * 1000 / 0.95:.2f} vs {plan_cost(1.5, 3, 42) * 1000 / 0.95:.2f}")
print(f"  保留与重建打平的等待 G = {(2 * 2 * 0.0000045 + 0.1 * 0.000014) / (2 * 0.0000045):.2f} s")
print(f"  Amdahl：工具加速 2 倍 {amdahl(0.1, 2):.2f}，模型加速 2 倍 {amdahl(0.9, 2):.2f}")

## 3. 代码里才看得到的

1. `call_cost` 把思考和可见输出都算进 `out`（生成计费），所以思考 1000 时 out = 1200。
2. `recovery_tree` 里"首次 → 修复 → 升级"的概率是 0.12 × 0.4 × 0.98 = 0.04704，条件概率按已经走到的节点算。
3. `plan_cost` 的模型费用用 6.79 $/卡时（Pod 价）；按量 8.64 $/卡时算，四个方案都会贵约 27%。

## 4. 自己改一改

- 给 `cache_worth` 加缓存过期：每 k 次访问要重新创建一次。
- 给 `recovery_tree` 加"修复失败直接返回失败"的策略，比较成本和按时率。
- 用 `plan_cost` 算模型调用 4 s 时，保留和重建哪个更省。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 打车起步价和里程省不掉，能砍的只是绕路；缓存第二趟就回本；
> 贵师傅一次做对，单件可能更便宜；
> 跑得多买车，跑得少打车，交点在单价之比；
> 缺考的人也占考场，按交卷人数分摊；
> 先按期限筛，再比按时成功的单价，最后看装不装得下。